# Module 4: LangSmith — Prompt Engineering, Observability & Evaluations

> Part of the **Modular Workshops** series. Standalone, ~30 min.

We cover four parts plus a closing loop:

1. **Prompt engineering** — author, test, and version prompts in the Playground and Prompt Hub, then pull them into code with the SDK.
2. **Tracing** — generate traces with the agent from Module 1, then query them with `list_runs` + filters.
3. **Offline evaluations** — build a dataset, then score two ways: in-process (final-response with LLM-as-judge, trajectory) and in a sandboxed environment with Harbor.
4. **Online evaluations** — score every new trace as it lands. Programmatic version + UI workflow.

Then **annotation queues** close the loop: route runs flagged by eval scores to a human for review.

<img src="../images/evals-conceptual.png" style="width: auto; max-height: 400px; border-radius: 8px;">

## Setup


In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.models import model
from utils.langsmith_rules import (
    get_or_create_annotation_queue,
    create_run_rule,
    delete_run_rule,
)

import os, time
from datetime import datetime, timedelta, timezone
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage
from langsmith import Client, uuid7

client = Client()
print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING", "not set"))
print("Project:", os.environ.get("LANGSMITH_PROJECT", "default"))


LANGSMITH_TRACING: true
Project: langsmith-workshop


## Part 1. Prompt Engineering — Playground & Prompt Hub

Before you can observe or evaluate an agent, you need a prompt worth shipping. LangSmith treats prompts as **versioned artifacts** — author and test them in the **Playground**, version and share them in the **Prompt Hub**, then pull them into code with the SDK. Three surfaces, one source of truth.

- **Playground** (UI) — an interactive editor: compose messages, wire up input variables, pick a model, and run.
- **Prompt Hub** (UI) — every saved prompt with full commit history, tags, and a public hub of community prompts to fork.
- **SDK** — `push_prompt` / `pull_prompt` to move prompts between code and the hub.

### 1.1 The Prompt Playground

Open **Prompts** in the LangSmith sidebar and click **+ Prompt** to land in the Playground. The left panel is your prompt — an ordered list of messages, each with a role:

- **System** — the instruction manual: persona and ground rules.
- **Human** — the user's turn.
- **AI** — a model turn, handy for few-shot examples.
- **Tool** — tool output, for testing how the model reacts to it.

Add an input variable by typing `{variable_name}` into any message (or highlight text and click **Convert to variable**). Fill in sample values in the right panel's **Inputs** box, then click **Start** to run and see the response.

**Template format.** Variables default to Python **f-string** syntax (`{topic}`). Switch to **mustache** (`{{topic}}`) from the format dropdown when you need loops, conditionals, or nested data (`{{user.name}}`) — f-strings only do flat substitution.

**Model configuration.** Click the **gear icon** next to the model name to set provider, model, temperature, and max tokens. Hit **Save As** to name a configuration — it's shared across your workspace and reusable in other LangSmith features.

**Tools.** Click **+ Tool** to attach tools: built-in ones (web search, code interpreter) or custom tools you define with a name, description, and argument schema. When the model calls a tool, the Playground shows the tool name and arguments so you can verify the call.

🔗 **Try it:** [Open Prompts in LangSmith →](https://smith.langchain.com/prompts) — then click **+ Prompt** (top right) to open the Playground.

### 1.2 Prompt Hub — save, version, share

Click **Save** in the Playground and your prompt lands in the **Prompts** table. Each prompt gets its own detail page with a two-pane layout: commit history and environments on the left, the selected commit on the right.

- **Commits** — every save is a new commit, and the full history is preserved. Toggle **Diff** (top-right) to compare a commit with its predecessor.
- **Tags** — mark a commit with a stable name (e.g. `prod`) so code can reference it without pinning a hash. Move or delete tags as the prompt evolves.
- **Environments** — reserved **Staging** and **Production** environments track which commit is live; **Promote** a commit to move it forward, or roll back from history.
- **Public hub** — search community prompts by name, use case, or model, and **fork** any of them into your workspace.

🔗 **Open in LangSmith:** [Your prompts →](https://smith.langchain.com/prompts) · [Public LangChain Hub →](https://smith.langchain.com/hub)

### 1.3 Manage prompts programmatically

Anything you do in the UI you can do from the SDK: `push_prompt` sends a prompt to the hub, and `pull_prompt` fetches it back. We'll do it in three quick steps — **push** a client-research prompt, **pull it and run it as an agent** (with `create_agent`, not a raw chain), then **version** it.

In [2]:
from langchain_core.prompts import ChatPromptTemplate

# Step 1 — author the assistant's system prompt and push it to the hub.
prompt_name = "client-research-assistant"
prompt = ChatPromptTemplate([
    ("system",
     "You are a client research assistant for a wealth management firm. "
     "Look up the client, then produce a concise pre-meeting briefing: "
     "(1) a one-line summary, (2) likely priorities and risk tolerance, "
     "(3) three tailored talking points, and (4) two questions the advisor should ask. "
     "Be factual and neutral; do not give specific investment advice."),
])

url = client.push_prompt(prompt_name, object=prompt)
print("Prompt page (click to open):", url)

Prompt page (click to open): https://smith.langchain.com/prompts/client-research-assistant/c2e69587?organizationId=fd6b1198-8e6a-4f06-80f5-20e1b40ded12


**Pull it back and run it — as an agent, not a chain.** `pull_prompt` returns the prompt we just pushed; we use it as the system prompt for `create_agent`, running on the workshop's shared `model`. A small mock `lookup_client` tool lets the agent fetch the profile, so the full agent loop (model → tool → model) shows up in the trace.

In [3]:
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def lookup_client(name: str) -> str:
    """Look up a wealth-management client's profile by name."""
    # Mock CRM lookup — swap for your real client data source in production.
    directory = {
        "Rohan Mehta": (
            "Rohan Mehta, 52, senior law-firm partner. ~$4.2M investable assets, "
            "moderately conservative. Two kids entering college within 3 years. "
            "Recently asked about tax-efficient withdrawals and ESG options."
        ),
    }
    return directory.get(name, f"No profile found for {name!r}.")


# Step 2 — pull the prompt back and run it with create_agent (uses the imported `model`).
pulled = client.pull_prompt(prompt_name)
system_prompt = pulled.format_messages()[0].content

agent = create_agent(model=model, tools=[lookup_client], system_prompt=system_prompt)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Prep me for my meeting with Rohan Mehta."}]}
)
print(result["messages"][-1].text)

## Pre-meeting briefing: Rohan Mehta

**1. One-line summary**  
Rohan Mehta, 52, is a senior law-firm partner with approximately $4.2M in investable assets, a moderately conservative profile, and near-term college funding and tax-efficiency concerns.

**2. Likely priorities and risk tolerance**  
- Funding two children’s college expenses within the next three years.  
- Managing tax-efficient withdrawals and cash-flow needs.  
- Exploring ESG-oriented investment options.  
- Moderately conservative risk tolerance, likely emphasizing preservation and liquidity around near-term obligations.

**3. Three tailored talking points**  
1. Review the expected timing and amount of college expenses, and how those liabilities fit within his broader financial plan.  
2. Discuss withdrawal sequencing, tax considerations, and the interaction with his income as a senior law-firm partner.  
3. Clarify what ESG means to him—screening, sustainability themes, impact, or portfolio integration—and how he ev

**Version it.** Re-push under the same name and LangSmith records a new commit — the earlier version stays in the history.

In [4]:
# Step 3 — re-push a tweaked version. Same name -> a new commit (full history preserved).
prompt_v2 = ChatPromptTemplate([
    ("system",
     "You are a client research assistant for a wealth management firm. "
     "Look up the client, then produce a concise pre-meeting briefing: "
     "(1) a one-line summary, (2) likely priorities and risk tolerance, "
     "(3) three tailored talking points, (4) two questions the advisor should ask, "
     "and (5) one risk or compliance flag to keep in mind. "
     "Be factual and neutral; do not give specific investment advice."),
])
url_v2 = client.push_prompt(prompt_name, object=prompt_v2)
print("New commit (click to open):", url_v2)

# Pull a specific commit with client.pull_prompt("client-research-assistant:<commit-hash>"),
# and tear down with client.delete_prompt("client-research-assistant") when you're done.

New commit (click to open): https://smith.langchain.com/prompts/client-research-assistant/18f8f53e?organizationId=fd6b1198-8e6a-4f06-80f5-20e1b40ded12


## Warm-up: Generate a few traces

Before we look at tracing and querying, let's actually produce some traces. 
We invoke the research agent from Module 1 (`agents/research_agent.py`) three times with **deliberately light** prompts — 
each one says "search at most once" so the runs finish in a few seconds instead of a few minutes.

On the trial run while building this module the warm-up took **~9 seconds total (3.1s avg per call)**. Expect similar.


In [5]:
from agents.research_agent import build_research_agent

agent = build_research_agent()

warmup_prompts = [
    "In one sentence, what is LangGraph for? Search at most once.",
    "In one sentence, what does the LangChain v1 release add? Search at most once.",
    "In one sentence, what is Tavily? Search at most once.",
]

total = 0.0
for q in warmup_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].text[:120]}")

print(f"\nTotal: {total:.1f}s ({total/len(warmup_prompts):.1f}s avg)")


[ 8.5s] LangGraph is for building and orchestrating stateful, long-running LLM agents and workflows.
[10.1s] LangChain v1 adds a focused `langchain` package centered on a new `create_agent` implementation built on LangGraph’s dur
[ 3.7s] Tavily is a web-access platform that provides AI applications with APIs for real-time search, content extraction, crawli

Total: 22.3s (7.4s avg)


## Part 2. Tracing + Querying Traces

Set `LANGSMITH_TRACING=true` and every LLM call, tool call, and state transition lands in your tracing project — no code changes required. 
(The warm-up above already produced traces; this section pulls them back out.)

We use `client.list_runs(...)` to query them.

In [6]:
project_name = os.environ.get("LANGSMITH_PROJECT", "modular-workshops")
try:
    project = client.read_project(project_name=project_name)
    print(f"Project: {project.name}")
    print(f"View traces: {project.url}")
except Exception as e:
    print(f"Could not read project (this is fine if first run): {e}")


Project: langsmith-workshop
View traces: https://smith.langchain.com/o/fd6b1198-8e6a-4f06-80f5-20e1b40ded12/projects/p/43879eb0-02fe-436b-a0fb-61a85d391d54


### 2.1 Pull recent traces

Useful filters on `client.list_runs(...)`:

- `project_name=` — scope to one project
- `start_time=` / `end_time=` — time window
- `run_type=` — `"llm"`, `"tool"`, `"chain"`, `"retriever"`
- `error=True` — only failed runs
- `is_root=True` — only top-level traces (not their children)
- `filter=` — LangSmith filter DSL (latency, feedback, attributes...)

In [7]:
from datetime import datetime, timedelta, timezone

# Pull the last hour of root traces from this workshop's project
since = datetime.now(timezone.utc) - timedelta(hours=1)

recent_runs = list(client.list_runs(
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    start_time=since,
    is_root=True,
    limit=20,
))

print(f"Found {len(recent_runs)} root run(s) in the last hour\n")
for r in recent_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds() if r.end_time else None
    print(f"- {r.id}  {r.name:25s}  latency={latency}s  error={r.error is not None}")


/var/folders/b0/x7cf1b5s1m3_wf4v8swpr_9r0000gn/T/ipykernel_20220/2500501162.py:6: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  recent_runs = list(client.list_runs(


Found 4 root run(s) in the last hour

- 01a11b84-8e9d-72c1-8e13-fca0d2eb6cb3  LangGraph                  latency=Nones  error=False
- 01a11b84-672c-7760-90a9-9ab669118089  LangGraph                  latency=10.095704s  error=False
- 01a11b84-45cd-7842-8131-1314ee602b43  LangGraph                  latency=8.538513s  error=False
- 01a11b84-2c03-7463-835d-5eefcfc5efdf  LangGraph                  latency=5.494874s  error=False


### 2.2 Filter DSL — find slow or errored runs

The `filter` argument is a small expression language. Common patterns:

- `gt(latency, 5)` — slower than 5 seconds
- `eq(status, "error")` — failed runs
- `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` — low-scored runs on a feedback key
- Combine with `and(...)` / `or(...)`

In [8]:
# Find slow root runs in the last hour (>5s latency)
slow_runs = list(client.list_runs(
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    start_time=since,
    is_root=True,
    filter='gt(latency, 5)',
    limit=20,
))

print(f"{len(slow_runs)} slow root run(s) (>5s) in the last hour")
for r in slow_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds()
    print(f"  {r.name:25s}  {latency:.1f}s  {r.id}")


/var/folders/b0/x7cf1b5s1m3_wf4v8swpr_9r0000gn/T/ipykernel_20220/526635509.py:2: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  slow_runs = list(client.list_runs(


3 slow root run(s) (>5s) in the last hour
  LangGraph                  10.1s  01a11b84-672c-7760-90a9-9ab669118089
  LangGraph                  8.5s  01a11b84-45cd-7842-8131-1314ee602b43
  LangGraph                  5.5s  01a11b84-2c03-7463-835d-5eefcfc5efdf


## Part 3. Offline Evaluations

**Offline evals** are the experiments you run on demand against a fixed dataset.
Build a dataset once, score your agent against it whenever you change a prompt, a model, or a tool — get a clean before/after comparison.

Three pieces:
1. **Dataset** — labeled `inputs` + expected `outputs`
2. **Target function** — runs your agent on each example
3. **Evaluators** — score the output (LLM-as-judge or code-based)

Two execution models, in that order:

- **3a. In-process** — call the agent directly and score what it returns.
- **3b. Environment evals** — run it in a sandboxed container and grade what it left on disk, including re-executing code it wrote.


### 3a. In-process evals

The agent runs in this process; each example's input maps to a value you score.


#### 3a.1 Dataset

Same input set, two reference shapes — one for final-response judging, one for trajectory matching.

In [9]:
examples = [
    {
        "inputs": {"query": "Write a haiku about Python to /haiku.txt"},
        "outputs": {
            "reference_answer": "A haiku about Python should be saved to /haiku.txt",
            "trajectory": ["write_file"],
        },
    },
    {
        "inputs": {"query": "Research the latest AI agent frameworks and write a brief report to /report.md"},
        "outputs": {
            "reference_answer": "A short report on recent AI agent frameworks, written to /report.md",
            "trajectory": ["task", "write_file"],
        },
    },
    {
        "inputs": {"query": "Write a short memo to /memo.md, then read it back to confirm"},
        "outputs": {
            "reference_answer": "A short memo written to /memo.md and read back",
            "trajectory": ["write_file", "read_file"],
        },
    },
    {
        "inputs": {"query": "Plan out a small research project on LangSmith, then research it and write a report to /report.md"},
        "outputs": {
            "reference_answer": "A planned-out research report on LangSmith written to /report.md",
            "trajectory": ["write_todos", "task", "write_file"],
        },
    },
]

dataset_name = "modular-workshops-evals"

if client.has_dataset(dataset_name=dataset_name):
    existing = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_id=existing.id)
    print(f"Deleted existing dataset '{dataset_name}'")

dataset = client.create_dataset(dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples],
    dataset_id=dataset.id,
)
print(f"Created dataset '{dataset_name}' with {len(examples)} examples")
print(f"View: {dataset.url}")


Deleted existing dataset 'modular-workshops-evals'
Created dataset 'modular-workshops-evals' with 4 examples
View: https://smith.langchain.com/o/fd6b1198-8e6a-4f06-80f5-20e1b40ded12/datasets/d679ed6f-40d4-4af4-936f-206603738592


#### 3a.2 Final-response eval (LLM-as-judge)

<img src="../images/final-response.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Treat the agent as a black box: did the final response satisfy the request? 
We'll build the **LLM-as-judge from scratch** so the moving parts are clear:

1. A Pydantic / TypedDict schema for the judge's output (`score`, `reasoning`)
2. A judge prompt that explains the grading criteria
3. `model.with_structured_output(...)` to force the LLM into the schema
4. An evaluator function that calls the judge and returns the score in the shape `client.evaluate` expects

In [10]:
def run_agent_final(inputs: dict) -> dict:
    """Run the agent and return its response plus any files it wrote.

    The evaluator receives these as separate fields so it can inspect both
    the assistant's message and the resulting file artifacts.
    """
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )

    return {
        "response": result["messages"][-1].text,
        "files": result.get("files") or {},
    }

In [11]:
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage

# Define the judge's output schema -- with_structured_output enforces this shape on the LLM response.
class CorrectnessGrade(TypedDict):
    """Score whether the agent's response satisfied the user's request."""
    score: bool   # True if correct/helpful, False otherwise
    reasoning: str  # one-sentence explanation

# The dataset's `reference_answer` is a SUCCESS RUBRIC, not an expected response text.
# Make that explicit to the judge so it doesn't downscore valid agent responses that
# happen to be worded differently.
correctness_judge_prompt = """You are an expert grader evaluating an AI assistant.

You will see:
1. The user's request
2. The assistant's final response
3. Any files written by the assistant
4. A success rubric

Evaluate both the response and the files. If the task asks the assistant to write a file, inspect the file contents when available. Mark `score=True` if the task was completed successfully, even if the wording differs from the rubric.

Mark `score=False` only if the task was clearly missed, the file is missing or incorrect, the response contains errors, or the assistant refused without good reason.

Give one short sentence of reasoning.
"""

# Bind the schema once -- `judge` is now a structured-output LLM.
judge = model.with_structured_output(CorrectnessGrade)

def correctness_evaluator(inputs, outputs, reference_outputs):
    grade = judge.invoke([
        SystemMessage(content=correctness_judge_prompt),
        HumanMessage(content=(
            f"User request: {inputs['query']}\n\n"
            f"Assistant response: {outputs['response']}\n\n"
            f"Files written: {outputs.get('files', {})}\n\n"
            f"Success rubric: {reference_outputs['reference_answer']}"
        )),
    ])

    return {
        "key": "correctness",
        "score": int(grade["score"]),
        "comment": grade["reasoning"],
    }

In [12]:
results = client.evaluate(
    run_agent_final,
    data=dataset_name,
    evaluators=[correctness_evaluator],
    experiment_prefix="final-response",
    max_concurrency=2,
)
print(f"View at: {results.experiment_name}")


View the evaluation results for experiment: 'final-response-6ac95445' at:
https://smith.langchain.com/o/fd6b1198-8e6a-4f06-80f5-20e1b40ded12/datasets/d679ed6f-40d4-4af4-936f-206603738592/compare?selectedSessions=77877717-0516-4b51-a0fb-123271388679




0it [00:00, ?it/s]

View at: final-response-6ac95445


#### 3a.3 Trajectory eval

<img src="../images/trajectory.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Score the **sequence of tool calls** the agent took, not just the final answer. Three evaluators:

- **`exact_match`** — did it take exactly the right steps in order?
- **`extra_steps`** — how many extra tool calls did it make?
- **`missing_steps`** — how many expected steps did it skip?

`extra_steps` and `missing_steps` use `collections.Counter` for multiset diffs — order doesn't matter for those two, but `exact_match` still catches ordering bugs.

In [13]:
from collections import Counter
from typing import Any

def trajectory_match(outputs, reference_outputs):
    return {
        "key": "exact_match",
        "score": int(outputs["trajectory"] == reference_outputs["trajectory"]),
    }

def extra_steps(outputs, reference_outputs):
    extras = Counter(outputs["trajectory"]) - Counter(reference_outputs["trajectory"])
    return {"key": "extra_steps", "score": sum(extras.values())}

def missing_steps(outputs, reference_outputs):
    missing = Counter(reference_outputs["trajectory"]) - Counter(outputs["trajectory"])
    return {"key": "missing_steps", "score": sum(missing.values())}


Next, we'll define the run function.

In [14]:
def extract_tool_calls(messages: list[Any]) -> list[str]:
    """Extract tool call names from messages in order."""
    tool_names = []
    for msg in messages:
        if getattr(msg, "tool_calls", None):
            tool_names.extend(tc["name"] for tc in msg.tool_calls)
    return tool_names

def run_agent_trajectory(inputs: dict) -> dict:
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )
    return {"trajectory": extract_tool_calls(result["messages"])}


In [15]:
results = client.evaluate(
    run_agent_trajectory,
    data=dataset_name,
    evaluators=[trajectory_match, extra_steps, missing_steps],
    experiment_prefix="trajectory",
    max_concurrency=2,
)
print(f"View at: {results.experiment_name}")


View the evaluation results for experiment: 'trajectory-22a96379' at:
https://smith.langchain.com/o/fd6b1198-8e6a-4f06-80f5-20e1b40ded12/datasets/d679ed6f-40d4-4af4-936f-206603738592/compare?selectedSessions=226c9187-c60f-411a-a82b-7a75c98461c8




0it [00:00, ?it/s]

View at: trajectory-22a96379


### 3b. Environment evals

An **environment eval** runs the agent in a sandbox and grades the files it produces.

In 3a, the agent runs as a function and returns a value for scoring. Here, the agent writes a report and a Python script. Verifying the report requires running that script, so the agent and verifier need an isolated process.

[Harbor](https://www.harborframework.com/) provides that isolation and manages the evaluation run.


Harbor uses five terms:

- **Task:** an instruction, an environment, and a verifier
- **Environment:** the sandbox in which the agent runs
- **Verifier:** the script that grades the agent's files
- **Trial:** one agent attempt at one task
- **Job:** a batch of trials across a set of tasks

Each directory under `environment_evals/dataset/` is one task. This workshop includes four topics that share the same structure and verifier.

```
noise-pollution-report/
├── task.toml                     <- timeouts, network policy, TOPIC_FILE
├── instruction.md                <- the prompt handed to the agent
├── environment/
│   ├── Dockerfile                <- builds the sandbox
│   └── info/*.md                 <- the seeded knowledge base
├── solution/solve.sh             <- oracle: sanity-checks the verifier
└── tests/
    ├── test.sh                   <- entry point Harbor runs after the agent stops
    └── verify.py                 <- grades the report, re-runs the agent's script
```

The workflow connects three pieces:

- **LangSmith sandboxes:** `--env langsmith` runs the container on LangSmith infrastructure instead of local Docker.
- **Deep Agents:** `--agent langgraph` loads a graph from `langgraph.json`.
- **LangSmith evaluations:** `--plugin langsmith` records the job as a dataset and experiment, the same objects used in 3a.

#### From evaluation to training

A programmatic verifier produces a **reward**. The same task format can support evaluation, reinforcement learning, or rejection-sampled fine-tuning. Harbor also records the full agent trajectory for each trial.

This example has two properties that matter for training:

- **Partial credit:** seven checks show which requirements passed instead of returning one binary result.
- **Multiple instances:** the four topics have different correct counts, so success depends on the procedure rather than one memorized answer. A training dataset would use many more variations; these four demonstrate the structure.

#### Distinct URLs, not occurrences

Each source file cites references both inline and in its `## Sources` section. The same URL therefore appears more than once, but the task asks for the number of **distinct** sources.

| `noise_pollution.md` | |
|---|---|
| URL occurrences | 7 |
| **Distinct** URLs | **3** |

- `len(re.findall(URL_RE, text))` returns **7** occurrences.
- `len(set(re.findall(URL_RE, text)))` returns **3** distinct URLs.

Both expressions are valid Python. Only the second matches the requirement.

#### 3b.1 The agent under test

This is the Module 1 research agent with one additional requirement: write `sources.py` so the source count can be reproduced. Harbor loads the graph from `environment_evals/deep-agent/agent.py` inside each trial container.

In [16]:
import json, subprocess
from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend

from utils.harbor_utils import seed_workspace, print_tree

REPORT_SYSTEM_PROMPT = """You are a research assistant that writes short reports from a local markdown knowledge base.

Your working directory has two folders:
  - `info/`   : source material, one markdown file per topic.
  - `report/` : where you MUST save the report.

Read the matching file in `info/`, then write a report to `report/` whose body is exactly
THREE paragraphs, followed by a `## Sources` section listing the reference URLs from that file.
If the request asks for a figure derived from the source material, put the calculation in
`sources.py` at the workspace root and run it. Use plain relative paths inside that script
(e.g. `info/topic.md`) so it still runs on its own.
"""

def build_report_agent():
    return create_deep_agent(
        model=model,
        system_prompt=REPORT_SYSTEM_PROMPT,
        backend=LocalShellBackend(root_dir=".", virtual_mode=False),
    )


First run the agent without Harbor. Seed a scratch workspace with the same files that the task's Dockerfile copies into its container.

In [17]:
task_dir = project_root / "environment_evals" / "dataset" / "noise-pollution-report"
workspace = seed_workspace(task_dir / "environment", project_root / ".dev-workspace")

instruction = (task_dir / "instruction.md").read_text()
run_config = {"configurable": {"thread_id": str(uuid7())}}

# The agent's backend is rooted at ".", so it has to run from inside the workspace.
os.chdir(workspace)
report_agent = build_report_agent()
agent_result = report_agent.invoke(
    {"messages": [{"role": "user", "content": instruction}]},
    config=run_config,
)
os.chdir(project_root / "modules")

reports = sorted((workspace / "report").glob("*.md"))
report_text = reports[0].read_text() if reports else "(none written)"

sources_py = workspace / "sources.py"
sources_text = sources_py.read_text() if sources_py.exists() else "(not written)"

print("=== report ===")
print(report_text)
print("\n=== sources.py ===")
print(sources_text)


=== report ===
Noise pollution is unwanted or harmful environmental sound generated by sources such as road traffic, aircraft, railways, industry, and construction. It is measured on a logarithmic decibel scale, and prolonged or intense exposure can damage hearing. Its effects accumulate across people and wildlife, making it especially consequential in dense urban areas.

The burden extends beyond hearing damage. Chronic environmental noise is associated with sleep disruption, elevated stress hormones, hypertension, and increased cardiovascular risk. Persistent exposure can also impair children’s reading comprehension and long-term memory, so noise functions as a quiet but significant public-health concern.

Reducing noise requires a combination of engineering, urban planning, and regulation. Noise barriers, quieter road surfaces, low-noise aircraft procedures, building insulation, and zoning that separates homes from industrial or transport corridors can reduce exposure. Strategic noi

#### 3b.2 The verifier

The verifier returns seven check scores and their average. Four checks cover report structure and citations. A fifth compares the count stated in the report with a value derived from the source file. The final two delete the report, rerun `sources.py`, and check its exit status and output.

This is an exact check, so it does not need an LLM judge.

The full verifier also handles path validation, file discovery, error handling, and result logging. This excerpt shows the checks and the generated-code execution boundary. See [`tests/verify.py`](../environment_evals/dataset/noise-pollution-report/tests/verify.py) for the executable implementation.

```python
NUMBER_WORDS = {
    1: "one", 2: "two", 3: "three", 4: "four", 5: "five", 6: "six",
    7: "seven", 8: "eight", 9: "nine", 10: "ten", 11: "eleven", 12: "twelve",
}

def paragraphs(body):
    """Return blank-line-separated prose blocks, excluding headings."""
    blocks = []
    for block in re.split(r"\n\s*\n", body.strip()):
        block = block.strip()
        if block and any(
            line.strip()
            for line in block.splitlines()
            if not line.strip().startswith("#")
        ):
            blocks.append(block)
    return blocks


def states_count(text, n):
    """Check whether a sentence about sources states n."""
    forms = [str(n)] + ([NUMBER_WORDS[n]] if n in NUMBER_WORDS else [])
    pattern = r"\b(" + "|".join(forms) + r")\b"
    return any(
        "source" in sentence.lower() and re.search(pattern, sentence, re.I)
        for sentence in re.split(r"(?<=[.!?])\s+|\n", text)
    )


files = report_files()
prose = read(files[0]) if files else ""
body, sources = split_body_and_sources(prose)

# Derive the expected answer from the source file, not the agent's output.
topic_path = resolve_under(INFO, TOPIC_FILE)
true_count = len(set(URL_RE.findall(read(topic_path))))

scores = {
    "report_written": float(bool(files)),
    "three_paragraphs": float(len(paragraphs(body)) == 3),
    "sources_section": float(bool(sources.strip())),
    "grounded_in_kb": float(any(url in prose for url in knowledge_base_urls())),
    "report_ties_to_code": float(states_count(prose, true_count)),
}

# Delete the report, then rerun the agent's script from a clean state.
for file in files:
    file.unlink(missing_ok=True)

completed = subprocess.run(
    ["python3", "sources.py"],
    cwd=str(WORKDIR),
    capture_output=True,
    text=True,
    timeout=60,
)
scores["script_reproduces"] = float(completed.returncode == 0)
scores["count_correct"] = float(
    bool(re.search(rf"\b{true_count}\b", completed.stdout))
)
scores["reward"] = round(sum(scores.values()) / len(scores), 4)
```


Harbor runs this verifier after the agent stops. `HARBOR_WORKDIR` lets us run the same file against the local scratch workspace:

In [18]:
verifier_env = {
    **os.environ,
    "HARBOR_WORKDIR": str(workspace),
    "HARBOR_LOGS": str(workspace / "logs"),
    "TOPIC_FILE": "noise_pollution.md",
}

completed = subprocess.run(
    ["python3", str(task_dir / "tests" / "verify.py")],
    env=verifier_env,
    capture_output=True,
    text=True,
)
print(completed.stdout)


{
  "report_written": 1.0,
  "three_paragraphs": 1.0,
  "sources_section": 1.0,
  "grounded_in_kb": 1.0,
  "report_ties_to_code": 1.0,
  "script_reproduces": 1.0,
  "count_correct": 1.0,
  "reward": 1.0
}



The verifier distinguishes between these two implementations:

| `sources.py` | counts | reward |
|---|---|---|
| `len(re.findall(URL_RE, text))` | every occurrence | **0.714** |
| `len(set(re.findall(URL_RE, text)))` | distinct URLs | **1.000** |

Both scripts run successfully, so both pass `script_reproduces`. Only the distinct-URL version passes `count_correct` and `report_ties_to_code`. If the four report-format checks and `script_reproduces` pass, the incorrect implementation scores `0.714`.

Recomputing the answer catches errors that are not visible from the finished report alone.


#### 3b.3 Packaging it as a Harbor task

`verify.py` contains the grading logic. The remaining files configure the task and its environment. The cells below display the files that Harbor reads from disk.

`task.toml` defines metadata, timeouts, and verifier environment variables. `TOPIC_FILE` lets the same verifier handle all four tasks:

In [19]:
print((task_dir / "task.toml").read_text())

schema_version = "1.4"
artifacts = []

[task]
name = "workshop/noise-pollution-report"
version = "1.0.0"
description = "Write a 3-paragraph sourced report from a local markdown knowledge base"
authors = []
keywords = []

[metadata]

[verifier]
timeout_sec = 300.0
collect = []

[verifier.env]
TOPIC_FILE = "noise_pollution.md"

[agent]
timeout_sec = 300.0

[environment]
network_mode = "public"
build_timeout_sec = 600.0
os = "linux"
mcp_servers = []

[environment.env]

[solution.env]
TOPIC_FILE = "noise_pollution.md"



`environment/Dockerfile` copies the knowledge base into the sandbox and creates an empty `report/` directory:

In [20]:
print((task_dir / "environment" / "Dockerfile").read_text())

# Base image with Python so Harbor can install and run the langgraph agent inside the
# container. The build context is this `environment/` directory.
FROM python:3.12-slim

WORKDIR /app

# Seed the research knowledge base and an empty report/ folder into the working directory.
# The agent reads from info/ and writes its report into report/; the verifier reads both.
COPY info/ /app/info/
RUN mkdir -p /app/report



`instruction.md` contains the task prompt and the reproducibility requirement:

In [21]:
print((task_dir / "instruction.md").read_text())

Write a report about **noise pollution**.

Choose the matching source file in `info/`, read it, and save a markdown report into
`report/` (name it `report/noise-pollution-report.md`). The body must be exactly three paragraphs,
followed by a `## Sources` section that carries over the reference URLs from the source file.

Your report must also state how many **distinct sources** the source file cites.

Every number in your report must be reproducible. Put the calculation in `sources.py` at the
workspace root and have it print that count.



The dataset contains four tasks. They differ only in their topic and source files:

In [22]:
print_tree(project_root / "environment_evals" / "dataset")

dataset/
  coffee-history-report/
    environment/
      Dockerfile
      info/
        coffee_history.md
        deep_sea_exploration.md
        noise_pollution.md
        urban_beekeeping.md
    instruction.md
    solution/
      solve.sh
    task.toml
    tests/
      test.sh
      verify.py
  deep-sea-report/
    environment/
      Dockerfile
      info/
        coffee_history.md
        deep_sea_exploration.md
        noise_pollution.md
        urban_beekeeping.md
    instruction.md
    solution/
      solve.sh
    task.toml
    tests/
      test.sh
      verify.py
  noise-pollution-report/
    environment/
      Dockerfile
      info/
        coffee_history.md
        deep_sea_exploration.md
        noise_pollution.md
        urban_beekeeping.md
    instruction.md
    solution/
      solve.sh
    task.toml
    tests/
      test.sh
      verify.py
  urban-beekeeping-report/
    environment/
      Dockerfile
      info/
        coffee_history.md
        deep_sea_exploration.md
    

#### 3b.4 Run it

Module 0's `uv sync` installs the `harbor` dependency group from `pyproject.toml`.

`--env langsmith` requires LangSmith sandboxes to be enabled for your workspace. If the environment cannot start, check that setting first.

Run the hand-written oracle before using a model. Every task should return `reward: 1.0`; otherwise the task or verifier is misconfigured.

In [23]:
!cd "{project_root}" && harbor run -p environment_evals/dataset -a oracle --env langsmith -y

⠙ 0/4 Running trials... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0:00:00 -:--:--
⠋ 0:00:00 noise-pollution-report__xqkPVzJ: starting environment... 
⠋ 0:00:00 coffee-history-report__xGR6Vf4: starting environment...  
⠋ 0:00:00 deep-sea-report__ZkjV3DH: starting environment...        
⠹ 0/4 Running trials... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0:00:00 -:--:--
⠙ 0:00:00 noise-pollution-report__xqkPVzJ: starting environment... 
⠙ 0:00:00 coffee-history-report__xGR6Vf4: starting environment...  
⠙ 0:00:00 deep-sea-report__ZkjV3DH: starting environment...        
⠸ 0/4 Running trials... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0:00:00 -:--:--
⠹ 0:00:00 noise-pollution-report__xqkPVzJ: starting environment... 
⠹ 0:00:00 coffee-history-report__xGR6Vf4: starting environment...  
⠹ 0:00:00 deep-sea-report__ZkjV3DH: starting environment...        
⠴ 0/4 Running trials... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0:00:00 -:--:--
⠸ 0:00:00 noise-pollution-report__xqkPVzJ: starting environment.

Next run the agent on all four tasks. A score below `1.0` shows which requirements failed.

The command reads `MODEL_SPEC`, so the container uses the model currently selected in `utils/models.py`.

In [24]:
# Model config comes from utils/models.py and travels through Harbor's config
# channel, so the container agent stays model-agnostic. The gateway is an
# OpenAI-compatible endpoint, so its key rides in OPENAI_API_KEY.
from utils.models import MODEL_SPEC, API_KEY_ENV

spec = dict(MODEL_SPEC)
model_name = spec.pop("model")

harbor_cmd = " ".join([
    f'cd "{project_root}" && harbor run',
    "-p environment_evals/dataset",
    "--agent langgraph",
    "--ak project_path=./environment_evals/deep-agent",
    "--ak graph=deep_agent",
    f"--model {model_name}",
    f"--ak model_kwargs='{json.dumps(spec)}'",
    f'--ae OPENAI_API_KEY="${API_KEY_ENV}"',
    "--env langsmith",
    "--plugin langsmith",
    "--pk dataset_name=environment-evals-reports",
    "--pk experiment_name=research-reports",
    "-y",
])
print(harbor_cmd)


cd "/Users/avirathkumar/langchain/demos/modular-workshops" && harbor run -p environment_evals/dataset --agent langgraph --ak project_path=./environment_evals/deep-agent --ak graph=deep_agent --model openai:gpt-5.6-luna --ak model_kwargs='{"base_url": "https://gateway.smith.langchain.com/openai", "use_responses_api": true}' --ae OPENAI_API_KEY="$LANGSMITH_API_KEY_GATEWAY" --env langsmith --plugin langsmith --pk dataset_name=environment-evals-reports --pk experiment_name=research-reports -y


Run the job:

In [25]:
!{harbor_cmd}


⠸ 0/4 Running trials... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0:00:00 -:--:--
⠴ 0/4 Running trials... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0:00:00 -:--:--
⠙ 0:00:00 noise-pollution-report__Zugkb8H: starting environment...
⠙ 0:00:00 deep-sea-report__b2U5jJx: running trial...              
⠙ 0:00:00 urban-beekeeping-report__F9D2iMa: running trial...      
⠦ 0/4 Running trials... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0:00:00 -:--:--
⠹ 0:00:00 noise-pollution-report__Zugkb8H: starting environment... 
⠹ 0:00:00 deep-sea-report__b2U5jJx: starting environment...        
⠹ 0:00:00 urban-beekeeping-report__F9D2iMa: starting environment...
⠧ 0/4 Running trials... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0:00:00 -:--:--
⠸ 0:00:00 noise-pollution-report__Zugkb8H: starting environment... 
⠸ 0:00:00 deep-sea-report__b2U5jJx: starting environment...        
⠸ 0:00:00 urban-beekeeping-report__F9D2iMa: starting environment...
⠏ 0/4 Running trials... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0:

The LangSmith plugin records the Harbor job as a project. Query its trials and scores with the same `list_runs` and `list_feedback` methods used in Part 2.

In [26]:
harbor_dataset = client.read_dataset(dataset_name="environment-evals-reports")
harbor_experiment = client.read_project(project_name="research-reports")

print(f"Dataset:    {harbor_dataset.url}")
print(f"Experiment: {harbor_experiment.url}\n")

for trial in client.list_runs(project_name="research-reports", is_root=True):
    feedback = client.list_feedback(run_ids=[trial.id])
    scores = {f.key: f.score for f in feedback}
    print(f"  {trial.name:34s} reward={scores.get('reward')}")


Dataset:    https://smith.langchain.com/o/fd6b1198-8e6a-4f06-80f5-20e1b40ded12/datasets/12af7a34-cf16-4a56-8b71-8f516b9f5eef
Experiment: https://smith.langchain.com/o/fd6b1198-8e6a-4f06-80f5-20e1b40ded12/projects/p/583e9c41-9306-4d57-89ab-f706554ce308



/var/folders/b0/x7cf1b5s1m3_wf4v8swpr_9r0000gn/T/ipykernel_20220/3954336488.py:7: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  for trial in client.list_runs(project_name="research-reports", is_root=True):


  urban-beekeeping-report__F9D2iMa   reward=1.0
  deep-sea-report__b2U5jJx           reward=1.0
  coffee-history-report__ER2rGbz     reward=0.2857
  noise-pollution-report__Zugkb8H    reward=1.0


Open **Datasets & Experiments** → `environment-evals-reports`. Each trial is a row and each of the seven reward keys is its own feedback column — the same surface 3a's evaluator scores land on. Same LangSmith objects, different execution model.

## Part 4. Online Evaluations

**Online evals** run automatically against every new trace as it lands in your tracing project — same evaluator as in Part 3, just triggered on incoming runs instead of a dataset.

LangSmith calls these **run rules**. The Python SDK doesn't expose them directly, so we wrap the REST endpoint with a helper at `utils/langsmith_rules.py`. 
Pass in: a project name, an LLM-as-judge prompt, an output schema. Get back: the rule ID and a deep link to inspect it in the UI.

In [28]:
# Define the LLM-as-judge prompt + schema.
judge_prompt = (
    "You score whether an assistant response satisfied the user's request.\n"
    "Reply with correctness (true/false) and one sentence of comment explaining why."
)

judge_schema = {
    "title": "correctness",
    "description": "Score whether the assistant response was correct/helpful.",
    "type": "object",
    "properties": {
        "correctness": {"type": "boolean", "description": "True if the response was correct/helpful"},
        "comment": {"type": "string", "description": "One short sentence explaining the score"},
    },
    "required": ["correctness", "comment"],
    "strict": True,
}

online_rule = create_run_rule(
    client,
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    display_name="workshop-online-correctness",
    sampling_rate=1.0,
    # Score only root traces, not every child LLM/tool/middleware span.
    filter="eq(is_root, true)",
    llm_judge_prompt=judge_prompt,
    llm_judge_schema=judge_schema,
)

print("Rule ID:", online_rule["id"])
print("Open in UI:", online_rule["url"])


Rule ID: ba376315-2994-4c0b-b755-07d25d967bb8
Open in UI: https://smith.langchain.com/o/fd6b1198-8e6a-4f06-80f5-20e1b40ded12/evaluators/5552a0d8-b6da-40c5-b612-d950589c9bad?ruleId=ba376315-2994-4c0b-b755-07d25d967bb8&sourceKind=session&sourceId=43879eb0-02fe-436b-a0fb-61a85d391d54


## Annotation Queues — Close the Loop

Once runs have **feedback scores** (from the online eval above, or any other source), route the low-scoring ones to a human for review.

LangSmith's annotation queues are that queue. We use **the same `create_run_rule` helper** — this time with `add_to_annotation_queue_id` set instead of an LLM judge. 
Any run matching the filter is added to the queue automatically.


In [29]:
queue = get_or_create_annotation_queue(
    client,
    name="modular-workshops-needs-review",
    description="Runs routed here by the workshop's correctness automation rule.",
)
print(f"Queue: {queue.name} (id={queue.id})")


Queue: modular-workshops-needs-review (id=fe70fd80-9818-4e4b-b9eb-79f2bba4ca7e)


In [30]:
# Same helper, no evaluator this time -- just a routing rule.
# Filter: only root traces (is_root=true) with correctness > 0.5.
queue_rule = create_run_rule(
    client,
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    display_name="workshop-route-correctness",
    sampling_rate=1.0,
    filter=(
        'and('
        'eq(is_root, true), '
        'eq(feedback_key, "correctness"), '
        'gt(feedback_score, 0.5)'
        ')'
    ),
    add_to_annotation_queue_id=queue.id,
)

print("Queue rule ID:", queue_rule["id"])
print("Open in UI:    ", queue_rule["url"])


Queue rule ID: 4bd6b361-8148-4a27-a3d4-869e9a266d27
Open in UI:     https://smith.langchain.com/o/fd6b1198-8e6a-4f06-80f5-20e1b40ded12/projects/p/43879eb0-02fe-436b-a0fb-61a85d391d54?runview=threads&tab=2


### Trigger both rules

Both rules are live. Run a few more light traces and you'll see:

1. The online eval fires on each new trace and attaches a `correctness` feedback score (~30s delay).
2. The queue rule fires on each *new feedback* that matches its filter (low correctness) and routes the run to the review queue.


In [ ]:
trigger_prompts = [
    "In one sentence, what is LangSmith? Search at most once.",
    "In one sentence, what does `create_agent` do in LangChain v1? Search at most once.",
    "In one sentence, what is the difference between LangChain and LangGraph? Search at most once.",
]

time.sleep(100)

total = 0.0
for q in trigger_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].text[:120]}")

# Use the tenant_id LangSmith returned with the rule so the link works regardless of workspace.
tenant_id = queue_rule["payload"]["tenant_id"]

print(f"\nTotal: {total:.1f}s.")
print(f"\nOnline eval rule:  {online_rule['url']}")
print(f"Queue rule:        {queue_rule['url']}")
print(f"Queue (review UI): https://smith.langchain.com/o/{tenant_id}/annotation-queues/{queue.id}")
print("\nFeedback shows up in the rule pages within ~30s; queue placements follow once feedback lands.")


[10.9s] LangSmith is a platform for developing, debugging, evaluating, and monitoring large language model (LLM) applications.
[ 8.8s] In LangChain v1, `create_agent` builds a configurable agent that combines a model, tools, prompts, and middleware into a
[10.2s] LangChain provides higher-level abstractions for building LLM applications and agents, while LangGraph is a lower-level 

Total: 30.0s.

Online eval rule:  https://smith.langchain.com/o/fd6b1198-8e6a-4f06-80f5-20e1b40ded12/evaluators/5552a0d8-b6da-40c5-b612-d950589c9bad?ruleId=ba376315-2994-4c0b-b755-07d25d967bb8&sourceKind=session&sourceId=43879eb0-02fe-436b-a0fb-61a85d391d54
Queue rule:        https://smith.langchain.com/o/fd6b1198-8e6a-4f06-80f5-20e1b40ded12/projects/p/43879eb0-02fe-436b-a0fb-61a85d391d54?runview=threads&tab=2
Queue (review UI): https://smith.langchain.com/o/fd6b1198-8e6a-4f06-80f5-20e1b40ded12/annotation-queues/fe70fd80-9818-4e4b-b9eb-79f2bba4ca7e

Feedback shows up in the rule pages within ~30s; queue pla

Failed to refresh cache entry client-research-assistant: Connection error caused failure to GET /commits/-/client-research-assistant/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/-/client-research-assistant/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 8] nodename nor servname provided, or not known)"))'))
Content-Length: None
API Key: lsv2_********************************************ed
Failed to refresh cache entry client-research-assistant: Connection error caused failure to GET /commits/-/client-research-assistant/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/-/client-re

### Common run-rule patterns

Swap the `filter` to build different rules:

| Use Case | `filter` |
|---|---|
| Low online correctness | `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` |
| Errored runs | `eq(status, "error")` |
| Slow runs | `gt(latency, 10)` |
| Long-running tool calls | `and(eq(run_type, "tool"), gt(latency, 3))` |
| Specific tool fired | `eq(name, "tavily_search")` |

Both rule types — online eval and queue routing — go through the same `create_run_rule` helper. 
Use `delete_run_rule(client, rule_id)` to tear them down when you're done.


## Recap

| Part | What | API |
|---|---|---|
| **1. Prompt engineering** | Author, version, and share prompts; pull them into code | `client.push_prompt(...)` / `client.pull_prompt(...)` |
| **Warm-up** | Generate a few traces with the Module 1 agent | `agent.invoke(...)` |
| **2. Tracing + querying** | Auto-capture every run; pull back by filter | `LANGSMITH_TRACING=true`, `client.list_runs(filter=...)` |
| **3a. Offline evals, in-process** | Score on demand against a dataset | `model.with_structured_output(...)` + `client.evaluate` |
| **3b. Offline evals, environment** | Run in a sandbox; grade what it left behind, re-running its own code | `harbor run --env langsmith --plugin langsmith` |
| **4. Online evals** | Score every new trace automatically | `create_run_rule(..., llm_judge_prompt=..., llm_judge_schema=...)` |
| **Annotation queues** | Route flagged runs for human review | `create_run_rule(..., add_to_annotation_queue_id=...)` |

The full loop: trace → online eval scores it → run rule routes low scores to the queue → human reviews → fixes flow into the next dataset.

**Next:** Module 5 — **Engine** automates this entire loop (detect → diagnose → PR → evaluator) on your deployed agent.